In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install folium

In [ ]:
!pip install contextily

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
import folium

# Konfigurasi plot agar lebih rapi
sns.set(style="whitegrid")

In [ ]:
import pandas as pd

#membaca file csv menggunakan pandas
df= pd.read_csv("/content/drive/MyDrive/Pratikum_Ml/Pratikum11/data/katalog_gempa.csv")


#cetak headet data (baris data) dari file
df.head()

In [ ]:
# Mengecek missing value
print(f"Jumlah Missing Value:\n{df.isnull().sum()}")

# Membersihkan data (drop rows dengan lat/lon kosong)
df_clean = df.dropna(subset=['lat', 'lon']).reset_index(drop=True)

# Mengecek duplikasi
print(f"Jumlah Duplikasi: {df_clean.duplicated().sum()}")
# Jika ada duplikasi persis, bisa dihapus untuk efisiensi, namun dalam gempa,
# koordinat sama bisa terjadi pada waktu berbeda (gempa susulan).
# Kita biarkan kecuali benar-benar duplikat seluruh kolom.

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df_clean, x='lon', y='lat', alpha=0.5, s=10, color='grey')
plt.title('Peta Persebaran Titik Gempa (Raw Data)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

In [ ]:
# Memilih fitur spasial
X = df_clean[['lat', 'lon']].values

# Inisialisasi Scaler
scaler = StandardScaler()

# Melakukan fitting dan transformasi
X_scaled = scaler.fit_transform(X)

# Menampilkan statistik hasil scaling (Mean mendekati 0, Std Dev = 1)
print(f"Mean setelah scaling: {np.mean(X_scaled, axis=0)}")
print(f"Std Dev setelah scaling: {np.std(X_scaled, axis=0)}")

In [ ]:
# Menentukan k = min_samples (umumnya 2 * dimensi = 4, kita pakai 5)
k = 5
nbrs = NearestNeighbors(n_neighbors=k).fit(X_scaled)
distances, indices = nbrs.kneighbors(X_scaled)

# Mengambil jarak ke tetangga terjauh (kolom terakhir) dan mengurutkannya
distance_desc = sorted(distances[:, k-1], reverse=False)

# Plotting K-Distance Graph
plt.figure(figsize=(10, 5))
plt.plot(distance_desc)
plt.title('K-Distance Graph (Elbow Method)')
plt.ylabel('Epsilon Distance (Scaled)')
plt.xlabel('Data Points sorted by distance')
plt.grid(True)
plt.show()

In [ ]:
# Inisialisasi dan fit DBSCAN
# eps=0.2 dan min_samples=5 adalah contoh, sesuaikan dengan hasil langkah 7
dbscan = DBSCAN(eps=0.2, min_samples=5)
clusters = dbscan.fit_predict(X_scaled)

# Menambahkan hasil cluster ke DataFrame asli
df_clean['cluster'] = clusters

# Menghitung statistik cluster
n_clusters_ = len(set(clusters)) - (1 if -1 in clusters else 0)
n_noise_ = list(clusters).count(-1)

print(f'Jumlah Cluster terbentuk: {n_clusters_}')
print(f'Jumlah Noise points: {n_noise_}')

In [ ]:
# Warning: Silhouette Score berat secara komputasi (O(N^2)).
# Jika data > 10.000 baris, gunakan sampel.

from sklearn.utils import resample
if len(X_scaled) > 10000:
    X_sample, labels_sample = resample(X_scaled, clusters, n_samples=5000, random_state=42)
    score = silhouette_score(X_sample, labels_sample)
else:
    score = silhouette_score(X_scaled, clusters)

print(f'Silhouette Score: {score:.3f}')

In [ ]:
plt.figure(figsize=(14, 8))

# 1. Plot Noise (Cluster -1) dengan warna pudar agar tidak mendistraksi
noise_data = df_clean[df_clean['cluster'] == -1]
plt.scatter(noise_data['lon'], noise_data['lat'],
            c='lightgrey', s=5, label='Noise', alpha=0.3)

# 2. Plot Cluster (Cluster != -1) dengan palet warna
cluster_data = df_clean[df_clean['cluster'] != -1]
scatter = plt.scatter(cluster_data['lon'], cluster_data['lat'],
                      c=cluster_data['cluster'], cmap='tab20',
                      s=10, alpha=1.0)

plt.title(f'Hasil Clustering DBSCAN (eps=0.2, min_samples=5)\nClusters: {n_clusters_}, Noise: {n_noise_}')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.colorbar(scatter, label='Cluster ID')
plt.legend()
plt.show()